# Single-Subject LORO Cache Test

This notebook mirrors the write-up workflow structure for one subject and writes subject-wise LORO `.npz` caches.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np

import sys
cwd = Path.cwd().resolve()
if (cwd / 'pyproject.toml').exists() and (cwd / 'src').exists():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'pyproject.toml').exists():
    REPO_ROOT = cwd.parent
elif (cwd.parent / 'pyproject.toml').exists():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ar_utils.run_loro_subject_cache import SubjectCacheConfig, list_eligible_subjects, process_subject_model


In [2]:
import importlib

ar_utils_run_loro_subject_cache = importlib.import_module("ar_utils.run_loro_subject_cache")
importlib.reload(ar_utils_run_loro_subject_cache)
SubjectCacheConfig = ar_utils_run_loro_subject_cache.SubjectCacheConfig
list_eligible_subjects = ar_utils_run_loro_subject_cache.list_eligible_subjects
process_subject_model = ar_utils_run_loro_subject_cache.process_subject_model

## 1. Configure run (analogous to write-up config cell)

In [3]:
CFG = SubjectCacheConfig(
    csv_path=str(REPO_ROOT / 'data' / 'raw' / 'gxp_samples.csv'),
    hvg_path=str(REPO_ROOT / 'data' / 'raw' / 'ahba_100hvg.txt'),
    out_root=str(REPO_ROOT / 'out' / 'loro_subject_cache'),
    gene_scope='hvg',
    use_cache=True,
    seed=123,
)

print(json.dumps(CFG.__dict__, indent=2))

{
  "csv_path": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/data/raw/gxp_samples.csv",
  "hvg_path": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/data/raw/ahba_100hvg.txt",
  "out_root": "/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache",
  "gene_scope": "hvg",
  "use_cache": true,
  "min_observed_parcels": 5,
  "c_min": 8,
  "n_comp_target": 3,
  "ridge_alpha_bridge": 0.01,
  "rbf_smoothing": 0.1,
  "gp_length_scale": 25.0,
  "gp_noise": 0.001,
  "seed": 123,
  "combat_use_covariates": true,
  "latent_dim": 3,
  "plam_max_iters": 5,
  "lambda_w": 1.0,
  "lambda_z": 1.0,
  "lambda_cal_a": 10.0,
  "lambda_cal_b": 10.0,
  "robust_loss": "student_t",
  "heteroscedastic": true,
  "calibration_mode": "hier_affine_map",
  "uncertainty_shrink": false
}


## 2. Select one eligible subject

In [4]:
eligible = list_eligible_subjects(CFG)
print('n_eligible:', len(eligible))
SUBJECT = 'GTEX-ZE7O' if 'GTEX-ZE7O' in eligible else eligible[0]
print('subject:', SUBJECT)

n_eligible: 313
subject: GTEX-ZE7O


In [5]:
eligible

['GTEX-1117F',
 'GTEX-111FC',
 'GTEX-117XS',
 'GTEX-1192X',
 'GTEX-11DXW',
 'GTEX-11DXY',
 'GTEX-11DYG',
 'GTEX-11DZ1',
 'GTEX-11EI6',
 'GTEX-11EMC',
 'GTEX-11GS4',
 'GTEX-11GSO',
 'GTEX-11GSP',
 'GTEX-11H98',
 'GTEX-11NUK',
 'GTEX-11NV4',
 'GTEX-11O72',
 'GTEX-11OF3',
 'GTEX-11ONC',
 'GTEX-11PRG',
 'GTEX-11TTK',
 'GTEX-11UD1',
 'GTEX-11WQC',
 'GTEX-11ZTS',
 'GTEX-11ZU8',
 'GTEX-11ZUS',
 'GTEX-11ZVC',
 'GTEX-12126',
 'GTEX-12WSA',
 'GTEX-12WSC',
 'GTEX-12WSD',
 'GTEX-12WSE',
 'GTEX-12WSF',
 'GTEX-12WSH',
 'GTEX-12WSM',
 'GTEX-12ZZW',
 'GTEX-12ZZX',
 'GTEX-12ZZY',
 'GTEX-12ZZZ',
 'GTEX-13112',
 'GTEX-1313W',
 'GTEX-131XH',
 'GTEX-131XW',
 'GTEX-131YS',
 'GTEX-132Q8',
 'GTEX-1399T',
 'GTEX-139T8',
 'GTEX-139TS',
 'GTEX-139TT',
 'GTEX-13CF2',
 'GTEX-13CZV',
 'GTEX-13FHO',
 'GTEX-13FHP',
 'GTEX-13FLV',
 'GTEX-13FLW',
 'GTEX-13FXS',
 'GTEX-13G51',
 'GTEX-13JUV',
 'GTEX-13JVG',
 'GTEX-13N1W',
 'GTEX-13N2G',
 'GTEX-13NYB',
 'GTEX-13NYS',
 'GTEX-13NZA',
 'GTEX-13O3O',
 'GTEX-13O3Q',
 'GTEX-13O

## 3. Build caches (naive, DLAM, PLAM)

In [6]:
print("Processing model: naive")
naive_path = process_subject_model(CFG, SUBJECT, 'naive')
print(f"Done: naive, path: {naive_path}")

Processing model: naive
[naive fold] GTEX-ZE7O fold=0 hold=5
[naive fold] GTEX-ZE7O fold=1 hold=12
[naive fold] GTEX-ZE7O fold=2 hold=18
[naive fold] GTEX-ZE7O fold=3 hold=38
[naive fold] GTEX-ZE7O fold=4 hold=109
[cache-write] naive GTEX-ZE7O -> /scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/naive/GTEX-ZE7O.npz
Done: naive, path: /scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/naive/GTEX-ZE7O.npz


In [7]:
print("Processing model: dlam")
dlam_path = process_subject_model(CFG, SUBJECT, 'dlam')
print(f"Done: dlam, path: {dlam_path}")

Processing model: dlam
[dlam fold] GTEX-ZE7O fold=0 hold=5
[dlam fold] GTEX-ZE7O fold=1 hold=12
[dlam fold] GTEX-ZE7O fold=2 hold=18
[dlam fold] GTEX-ZE7O fold=3 hold=38
[dlam fold] GTEX-ZE7O fold=4 hold=109
[cache-write] dlam GTEX-ZE7O -> /scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/dlam/GTEX-ZE7O.npz
Done: dlam, path: /scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/dlam/GTEX-ZE7O.npz


In [8]:
print("Processing model: plam")
plam_path = process_subject_model(CFG, SUBJECT, 'plam')
print(f"Done: plam, path: {plam_path}")

Processing model: plam
[plam fold] GTEX-ZE7O fold=0 hold=5
[plam fold] GTEX-ZE7O fold=1 hold=12
[plam fold] GTEX-ZE7O fold=2 hold=18
[plam fold] GTEX-ZE7O fold=3 hold=38
[plam fold] GTEX-ZE7O fold=4 hold=109
[cache-write] plam GTEX-ZE7O -> /scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/plam/GTEX-ZE7O.npz
Done: plam, path: /scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/plam/GTEX-ZE7O.npz


In [10]:
paths = {
    'naive': naive_path,
    'dlam': dlam_path,
    'plam': plam_path,
}
paths

{'naive': PosixPath('/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/naive/GTEX-ZE7O.npz'),
 'dlam': PosixPath('/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/dlam/GTEX-ZE7O.npz'),
 'plam': PosixPath('/scratch/asr655/neuroinformatics/Seq2GeneEx/gtex_gp/out/loro_subject_cache/hvg/plam/GTEX-ZE7O.npz')}

## 4. Inspect NPZ payload and masks

In [12]:
for model, path in paths.items():
    payload = np.load(path, allow_pickle=True)
    print('\nMODEL:', model)
    print('predictions_subject_h', payload['predictions_subject_h'].shape)
    print('truth_loro_h', payload['truth_loro_h'].shape)
    print('gtex_mask sum', int(payload['gtex_mask'].sum()))
    print('loro_eval_mask sum', int(payload['loro_eval_mask'].sum()))
    print('imputed_mask sum', int(payload['imputed_mask'].sum()))


MODEL: naive
predictions_subject_h (150, 88)
truth_loro_h (150, 88)
gtex_mask sum 11
loro_eval_mask sum 5
imputed_mask sum 145

MODEL: dlam
predictions_subject_h (150, 88)
truth_loro_h (150, 88)
gtex_mask sum 11
loro_eval_mask sum 5
imputed_mask sum 145

MODEL: plam
predictions_subject_h (150, 88)
truth_loro_h (150, 88)
gtex_mask sum 11
loro_eval_mask sum 5
imputed_mask sum 145


## 5. Next

- These subject files can be aggregated by a reducer script into cohort-level tensors/metrics.
- Re-running this notebook should hit cache quickly when config and source signatures are unchanged.